# Lab 6: The Iterative Workflow — Diagnose, Hypothesise, Test

**Before you start: go to Runtime → Change runtime type and select GPU.**

Your project proposal is handed in. From now on, most of your time will be spent going around
this loop (from the project introduction slides):

```
        ┌──────────────► 1. Clarification ─────────────┐
        │               list all issues with           ▼
 5. Evaluation          your current setup      2. Prioritization
 test one or more                               select the most
 of the solutions                               critical issue
        ▲                                              │
        │                                              ▼
 4. Design solutions ◄──────────────────────── 3. Analysis
 hypothesise possible                          find possible causes
 solutions to the issue                        of the selected issue
```

The loop only works if each step is backed by **evidence**. You need to be able to trust your
pipeline, to tell a real improvement from random noise, to see *why* a model is failing, and to
change one thing at a time. That is what this lab is about.

| Part | What you do | Where it sits in the loop |
|---|---|---|
| 1. Bug hunt | Use four sanity checks to find bugs in a broken pipeline | Before the loop: can you trust your setup? |
| 2. Baseline and noise floor | Run the baseline with 3 seeds and start an experiment log | Initial stage |
| 3. Learning-rate sweep | Find a working learning rate and read the loss curves | Clarification |
| 4. A worked iteration | A deep CNN fails to train → diagnose with activation histograms and gradient flow → test two hypotheses | The full loop |
| 5. Ablation | Remove components from the final model, one at a time | Evaluation |
| 6. Budget | How many experiments can you afford before December? | Prioritization |
| 7. *Optional:* Grad-CAM | Is the model looking at the right thing? | Analysis |

Parts 1–5 use **SVHN** and the small CNN from Lab 4a, so the notebook runs quickly and you already
know the setup. Everything in Parts 1, 2, 4 and 5 (sanity checks, seeds, activation/gradient
diagnostics, ablations) carries over directly to segmentation and detection projects.

**A habit for today:** every time you see ✍️ **Prediction**, write down what you expect *before*
running the next cell. It takes 30 seconds, and it is the difference between testing a hypothesis
and just trying things. A wrong prediction is not a failure — it is usually the most interesting
thing you learn that day.

After the notebook, start your **iteration log** (`Lab6_IterationLog.md`) with your group. This
is a running document you will keep adding to for the rest of the course.

## Setup

In [ ]:
import copy
import math
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


def set_seed(seed):
    '''Makes weight initialisation and batch order reproducible.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

### The data

We use a **10,000-image training subset** and a **2,000-image validation subset** of the SVHN
training split. We don't touch the test split at all today — every decision below is made on the
validation set.

Using a subset is itself a workflow technique: it makes each run take seconds instead of minutes,
so you can run many more experiments. More on that in Part 6.

For speed we load the whole subset into GPU memory once and do batching ourselves (see `batches`
below). In your own project you will normally use a `DataLoader` as in Lab 3 — the workflow is the
same either way.

In [ ]:
N_TRAIN, N_VAL = 10_000, 2_000
BATCH_SIZE = 64
NUM_CLASSES = 10

svhn = torchvision.datasets.SVHN(root='./data', split='train', download=True)
# svhn.data is a uint8 numpy array of shape (N, 3, 32, 32); svhn.labels holds the digits 0-9

perm = torch.randperm(len(svhn), generator=torch.Generator().manual_seed(0))
train_idx = perm[:N_TRAIN]
val_idx = perm[N_TRAIN:N_TRAIN + N_VAL]


def load_subset(idx):
    x = torch.from_numpy(svhn.data[idx.numpy()]).float() / 255.0   # (N, 3, 32, 32) in [0, 1]
    y = torch.from_numpy(np.asarray(svhn.labels)[idx.numpy()]).long()
    return x, y


x_train, y_train = load_subset(train_idx)
x_val, y_val = load_subset(val_idx)

# Normalisation statistics come from the TRAINING set only, then get applied to both splits.
MEAN = x_train.mean(dim=(0, 2, 3), keepdim=True)
STD = x_train.std(dim=(0, 2, 3), keepdim=True)
x_train = ((x_train - MEAN) / STD).to(device)
x_val = ((x_val - MEAN) / STD).to(device)
y_train, y_val = y_train.to(device), y_val.to(device)

print(f'Train: {tuple(x_train.shape)}, Val: {tuple(x_val.shape)}')
print('Train class counts:', torch.bincount(y_train, minlength=NUM_CLASSES).tolist())

In [ ]:
def random_shift(xb, max_shift=4):
    '''Data augmentation: shift each image by up to `max_shift` pixels (like RandomCrop with padding).'''
    n, _, h, w = xb.shape
    shifts = torch.randint(-max_shift, max_shift + 1, (n, 2), device=xb.device).float()
    theta = torch.zeros(n, 2, 3, device=xb.device)
    theta[:, 0, 0] = 1
    theta[:, 1, 1] = 1
    theta[:, 0, 2] = shifts[:, 0] * 2 / w
    theta[:, 1, 2] = shifts[:, 1] * 2 / h
    grid = F.affine_grid(theta, xb.shape, align_corners=False)
    return F.grid_sample(xb, grid, padding_mode='zeros', align_corners=False)


def batches(x, y, batch_size=BATCH_SIZE, shuffle=True, augment=False):
    '''Yields (images, labels) mini-batches from tensors that already live on the GPU.'''
    order = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for i in range(0, len(x), batch_size):
        idx = order[i:i + batch_size]
        xb, yb = x[idx], y[idx]
        if augment:
            xb = random_shift(xb)
        yield xb, yb

### Models

`ToyCNN` is the Lab 4a template with the gaps filled in. `DeepCNN` is a deeper plain network
(9 conv layers, no skip connections) that we will need in Part 4. Both let you choose the
**weight initialisation**, and `DeepCNN` can optionally use **batch normalisation**, both from
this week's lecture.

In [ ]:
class ToyCNN(nn.Module):
    '''The Lab 4a architecture: 3 conv layers + 2 linear layers.'''
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding='same'), nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3), nn.ReLU(),            # 32x32 -> 30x30
            nn.MaxPool2d(2, 2),                                     # -> 15x15
            nn.Conv2d(32, 64, kernel_size=3, padding='same'), nn.ReLU(),
            nn.MaxPool2d(2, 2),                                     # -> 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, num_classes),                            # raw logits, no softmax
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class DeepCNN(nn.Module):
    '''A deeper plain CNN: 3 stages x 3 conv layers, optional batch norm, no skip connections.'''
    def __init__(self, num_classes=NUM_CLASSES, widths=(32, 64, 128), convs_per_stage=3, batchnorm=False):
        super().__init__()
        layers, in_ch = [], 3
        for out_ch in widths:
            for _ in range(convs_per_stage):
                # A conv bias is redundant when BN follows (BN has its own shift, beta)
                layers.append(nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=not batchnorm))
                if batchnorm:
                    layers.append(nn.BatchNorm2d(out_ch))
                layers.append(nn.ReLU())
                in_ch = out_ch
            layers.append(nn.MaxPool2d(2, 2))                       # 32 -> 16 -> 8 -> 4
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(widths[-1] * 4 * 4, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def init_weights(model, scheme):
    '''scheme: "default" (PyTorch's built-in), "small" (N(0, 0.01^2)), or "kaiming" (He et al.).'''
    if scheme == 'default':
        return
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            if scheme == 'small':
                nn.init.normal_(m.weight, mean=0.0, std=0.01)
            elif scheme == 'kaiming':
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            else:
                raise ValueError(scheme)
            if m.bias is not None:
                nn.init.zeros_(m.bias)


def build_model(arch='toy', init='default', batchnorm=False):
    model = ToyCNN() if arch == 'toy' else DeepCNN(batchnorm=batchnorm)
    init_weights(model, init)
    return model.to(device)

### The training function and the experiment log

Every run goes through `run_experiment`. It takes a **configuration** (everything that defines
the experiment) and appends one row to `EXPERIMENT_LOG`, which is also saved to
`experiment_log.csv` after every run.

This is the most boring and most useful cell in the notebook. In December, when you write your
Results section, you will want a table of *every* run with its exact settings — not a memory of
"I think the one with BN was better". Set up something like this for your own project this week
(a CSV file is enough; [Weights & Biases](https://wandb.ai) is a nice upgrade).

**Held fixed across all experiments today:** the data split, SGD with momentum 0.9 (momentum
is covered in the next lecture; treat it as a fixed choice for now), batch size 64, and the
evaluation code. Anything not in the config is part of the controlled setup.

In [ ]:
EXPERIMENT_LOG = []
loss_fn = nn.CrossEntropyLoss()


@torch.no_grad()
def evaluate(model, x, y, batch_size=512):
    model.eval()
    total_loss, correct = 0.0, 0
    for xb, yb in batches(x, y, batch_size=batch_size, shuffle=False):
        logits = model(xb)
        total_loss += loss_fn(logits, yb).item() * len(xb)
        correct += (logits.argmax(1) == yb).sum().item()
    return total_loss / len(x), correct / len(x)


def run_experiment(name, arch='toy', init='default', batchnorm=False, augment=False,
                   lr=0.01, epochs=10, seed=0, log=True, verbose=True):
    '''Trains one model from scratch and logs the result. Returns (model, history).'''
    config = dict(name=name, arch=arch, init=init, batchnorm=batchnorm, augment=augment,
                  lr=lr, epochs=epochs, seed=seed, batch_size=BATCH_SIZE)
    set_seed(seed)
    model = build_model(arch, init, batchnorm)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)

    history = {'iter_loss': [], 'train_loss': [], 'val_loss': [], 'val_acc': []}
    diverged = False
    start = time.time()
    for epoch in range(epochs):
        model.train()
        running, seen = 0.0, 0
        for xb, yb in batches(x_train, y_train, augment=augment):
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            if not torch.isfinite(loss):
                diverged = True
                break
            history['iter_loss'].append(loss.item())
            running += loss.item() * len(xb)
            seen += len(xb)
        if diverged:
            if verbose:
                print(f'  [{name} | seed {seed}] diverged (loss = {loss.item()}) in epoch {epoch + 1}')
            break
        val_loss, val_acc = evaluate(model, x_val, y_val)
        history['train_loss'].append(running / seen)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
    runtime = time.time() - start

    result = dict(config,
                  diverged=diverged,
                  train_loss=history['train_loss'][-1] if history['train_loss'] else float('nan'),
                  val_loss=history['val_loss'][-1] if history['val_loss'] else float('nan'),
                  val_acc=history['val_acc'][-1] if history['val_acc'] else float('nan'),
                  runtime_s=round(runtime, 1))
    if log:
        EXPERIMENT_LOG.append(result)
        pd.DataFrame(EXPERIMENT_LOG).to_csv('experiment_log.csv', index=False)
    if verbose and not diverged:
        print(f'  [{name} | seed {seed}] train loss {result["train_loss"]:.3f} | '
              f'val loss {result["val_loss"]:.3f} | val acc {result["val_acc"]:.3f} | {runtime:.0f}s')
    return model, history


def summarise(names):
    '''Mean ± std over seeds for the named experiments, from the log.'''
    df = pd.DataFrame(EXPERIMENT_LOG)
    df = df[df['name'].isin(names)]
    out = df.groupby('name', sort=False).agg(runs=('seed', 'count'),
                                             val_acc_mean=('val_acc', 'mean'),
                                             val_acc_std=('val_acc', 'std'),
                                             train_loss_mean=('train_loss', 'mean'),
                                             runtime_s=('runtime_s', 'mean'))
    return out.reindex(names).round(4)

---
## Part 1: Bug hunt — can you trust your pipeline?

Deep learning code rarely crashes when it's wrong. It trains, prints numbers, produces a
plausible-looking loss curve — and gives you a model that is quietly worse than it should be.
If you start comparing experiments on top of a buggy pipeline, every conclusion you draw is
suspect.

Below is a pipeline written by a (fictional) previous student. It runs without errors.
**It contains four bugs.** Your job is to find and fix them *using the four sanity checks
below*, not by reading the code line by line. (In your project, the bug is never in the
place you are looking.)

### The four sanity checks

| Check | What you expect | What it catches |
|---|---|---|
| **A. Look at the data** | Images look like digits; after normalisation each channel has mean ≈ 0, std ≈ 1 in *both* splits | Preprocessing errors, wrong normalisation, train/val preprocessed differently |
| **B. Check the split** | No example appears in both train and validation | Leakage (overly optimistic validation results) |
| **C. Initial loss** | With random weights and C balanced classes, the model should be roughly uniform: loss ≈ −ln(1/C) = ln(C). For C = 10 that's **2.30** | Bad preprocessing, broken initialisation, wrong loss setup |
| **D. Overfit one batch** | Train repeatedly on a single batch of 32 images. Loss should go to ≈ 0 and accuracy to 100% within a few hundred steps | Almost any bug in the model, loss or training step. A model that can't memorise 32 images can't learn anything |

**Why one batch and not one image?** A single image can be memorised even when things are badly
broken (e.g., labels misaligned with images — the model just learns the one label). 32 different
images with 10 different labels is a much stronger test, and still takes only seconds.

*Adapting this to your project:* A, B and D apply to every task. For segmentation, "overfit one
batch" means the predicted masks should match the ground truth almost perfectly; for detection,
the boxes should. For C: per-pixel cross-entropy in segmentation also starts near ln(C); for
regression, compare the initial loss to the loss of always predicting the mean target.

### The buggy pipeline

Run this cell as it is. Don't fix anything yet.

In [ ]:
# ======================== BUGGY PIPELINE — DO NOT TRUST ========================

def buggy_data():
    g = torch.Generator().manual_seed(1)
    p = torch.randperm(len(svhn), generator=g)
    tr_idx = p[:N_TRAIN]
    va_idx = p[N_TRAIN - N_VAL // 4:N_TRAIN + 3 * N_VAL // 4]
    x_tr = torch.from_numpy(svhn.data[tr_idx.numpy()]).float()
    x_va = torch.from_numpy(svhn.data[va_idx.numpy()]).float()
    y_tr = torch.from_numpy(np.asarray(svhn.labels)[tr_idx.numpy()]).long()
    y_va = torch.from_numpy(np.asarray(svhn.labels)[va_idx.numpy()]).long()
    return tr_idx, va_idx, x_tr.to(device), y_tr.to(device), x_va.to(device), y_va.to(device)


class BuggyCNN(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding='same'), nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding='same'), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def buggy_train_step(model, optimizer, xb, yb):
    logits = model(xb)
    loss = loss_fn(logits, yb)
    loss.backward()
    optimizer.step()
    return loss.item(), (logits.argmax(1) == yb).float().mean().item()

# ==============================================================================

b_train_idx, b_val_idx, bx_train, by_train, bx_val, by_val = buggy_data()
print('Buggy pipeline loaded.')

### The checks

These four functions are the sanity checks. They are written to be reusable: copy them into your
own project.

In [ ]:
def check_data(x_tr, y_tr, x_va, mean=None, std=None, n_show=8):
    '''Check A: per-channel statistics of both splits + a look at some training images.'''
    for split, x in [('train', x_tr), ('val', x_va)]:
        m = x.mean(dim=(0, 2, 3)).tolist()
        s = x.std(dim=(0, 2, 3)).tolist()
        print(f'{split:>5}: mean per channel = {[round(v, 3) for v in m]}, '
              f'std per channel = {[round(v, 3) for v in s]}, '
              f'min = {x.min().item():.2f}, max = {x.max().item():.2f}')
    imgs = x_tr[:n_show].detach().cpu()
    if mean is not None:                       # undo normalisation for display
        imgs = imgs * std + mean
    imgs = (imgs - imgs.amin(dim=(1, 2, 3), keepdim=True)) / \
           (imgs.amax(dim=(1, 2, 3), keepdim=True) - imgs.amin(dim=(1, 2, 3), keepdim=True) + 1e-8)
    fig, axes = plt.subplots(1, n_show, figsize=(1.6 * n_show, 2))
    for ax, img, label in zip(axes, imgs, y_tr[:n_show].tolist()):
        ax.imshow(img.permute(1, 2, 0).numpy())
        ax.set_title(str(label))
        ax.axis('off')
    plt.suptitle('Training images with their labels (display rescaled per image)')
    plt.tight_layout()
    plt.show()


def check_split(tr_idx, va_idx):
    '''Check B: are any examples in both splits?'''
    overlap = set(tr_idx.tolist()) & set(va_idx.tolist())
    print(f'Train size {len(tr_idx)}, val size {len(va_idx)}, overlap: {len(overlap)} examples')
    if overlap:
        print(f'WARNING: {len(overlap) / len(va_idx):.0%} of the validation set is also in the training set!')


@torch.no_grad()
def check_initial_loss(model, x, y, num_classes=NUM_CLASSES):
    '''Check C: loss of the untrained model vs. the expected ln(C).'''
    model.eval()
    loss = loss_fn(model(x[:512]), y[:512]).item()
    print(f'Initial loss: {loss:.3f}   (expected ≈ ln({num_classes}) = {math.log(num_classes):.3f})')


def overfit_one_batch(model, x, y, train_step, steps=300, lr=0.01, batch_size=32):
    '''Check D: can the model memorise a single small batch?'''
    xb, yb = x[:batch_size], y[:batch_size]
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    model.train()
    losses, accs = [], []
    for _ in range(steps):
        loss, acc = train_step(model, optimizer, xb, yb)
        losses.append(loss)
        accs.append(acc)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.2))
    ax1.plot(losses)
    ax1.set_yscale('log')
    ax1.set_xlabel('Step'); ax1.set_ylabel('Training loss (log scale)'); ax1.set_title('Loss on one batch')
    ax1.grid(alpha=0.3)
    ax2.plot(accs)
    ax2.set_ylim(-0.02, 1.02)
    ax2.set_xlabel('Step'); ax2.set_ylabel('Accuracy'); ax2.set_title('Accuracy on the same batch')
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f'Final loss: {losses[-1]:.4f}, final accuracy: {accs[-1]:.2f}')

### Run the checks on the buggy pipeline

✍️ **Prediction:** before you run the next cell, what should the output look like if the pipeline
were correct?

In [ ]:
print('--- Check A: data ---')
check_data(bx_train, by_train, bx_val)

print('\n--- Check B: split ---')
check_split(b_train_idx, b_val_idx)

print('\n--- Check C: initial loss ---')
set_seed(0)
check_initial_loss(BuggyCNN().to(device), bx_train, by_train)

print('\n--- Check D: overfit one batch ---')
set_seed(0)
overfit_one_batch(BuggyCNN().to(device), bx_train, by_train, buggy_train_step)

### Your task

1. For each check that fails, write down what you observe and what might cause it.
2. Fix **one bug at a time** in the buggy pipeline cell, re-run it, then re-run the checks. Fixing one
   bug often changes what the other checks show — for instance, a data bug can hide a model bug.
3. Continue until all four checks pass: sensible statistics, no overlap, initial loss ≈ 2.30, and
   one-batch loss → ≈ 0 with accuracy 1.0.

Only open the hints if you're stuck.

<details>
<summary>Hint for Check A</summary>

What range are the pixel values in? Compare with how `load_subset` and the `MEAN`/`STD` lines
prepare the data at the top of the notebook.
</details>

<details>
<summary>Hint for Check B</summary>

Look at how `va_idx` is sliced from `p`.
</details>

<details>
<summary>Hint for Check C</summary>

Once the data is fixed, the initial loss is about 4.6. Which number has a natural logarithm of 4.6?
How many outputs does `BuggyCNN` have? (This one is sneaky: the model still trains and can reach a
reasonable accuracy, because the 90 extra outputs just learn to be very negative.)
</details>

<details>
<summary>Hint for Check D</summary>

Compare `buggy_train_step` line by line with the training loop in `run_experiment`. What
happens to the gradients from the previous step?
</details>

**Discussion:** which of these bugs would still have given you a model with a "reasonable"
validation accuracy? Which would have made your validation accuracy *better* than it should be?

---
## Part 2: The baseline and the noise floor

From now on we use the correct pipeline from the top of the notebook.

Suppose you train your baseline and get 85.2% validation accuracy. Then you add a change and get
86.0%. Did the change help?

You can't tell without knowing how much the result varies when **nothing** changes. Training is
random: the initial weights and the order of the mini-batches depend on the random seed. So
let's run the same baseline three times with different seeds.

✍️ **Prediction:** how much do you expect the validation accuracy to vary across three seeds?
0.01%? 0.1%? 1%? More?

In [ ]:
BASELINE_LR = 0.01
for seed in [0, 1, 2]:
    run_experiment('E0_baseline', arch='toy', lr=BASELINE_LR, seed=seed)

summarise(['E0_baseline'])

**What varies between seeds here:** weight initialisation and batch order. What doesn't: the
train/validation split (fixed by `manual_seed(0)` when we loaded the data). In projects with small
validation sets, the *choice of split* often causes even more variation than the seed — if your
validation set is small, consider repeating with different splits (cross-validation, Lecture 2).

**A rule of thumb for your project:** run every configuration you compare with **at least 3 seeds**
and report mean ± standard deviation. A difference smaller than about two standard deviations is
not evidence of anything. If runs are too expensive for that, say so in your report and treat
small differences as inconclusive.

Now look at the experiment log. This is the table your Results section will eventually be built from.

In [ ]:
pd.DataFrame(EXPERIMENT_LOG)

---
## Part 3: A learning-rate sweep

The learning rate is almost always the first hyperparameter to get right. If it's wrong,
every other comparison you make is distorted. (You saw in Lecture 2 what happens when it's too
high or too low. The next lecture covers optimisers and learning-rate schedules in depth.)

A sweep tries learning rates spaced on a **log scale** — 1e-4, 1e-3, 1e-2, … — because the useful
range spans several orders of magnitude. We use short runs (3 epochs, 1 seed): the goal is to
find the right *region*, not the perfect value. This is a **screening** experiment.

✍️ **Prediction:** sketch (on paper) the training-loss curve you expect for a learning rate that is
far too low, about right, and far too high.

In [ ]:
SWEEP_LRS = [1e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0]
sweep_histories = {}
for lr in SWEEP_LRS:
    _, hist = run_experiment(f'sweep_lr={lr:g}', arch='toy', lr=lr, epochs=3, seed=0)
    sweep_histories[lr] = hist

In [ ]:
def smooth(values, k=10):
    if len(values) < k:
        return np.asarray(values)
    return np.convolve(values, np.ones(k) / k, mode='valid')


colors = plt.cm.viridis(np.linspace(0.05, 0.9, len(SWEEP_LRS)))   # ordered values -> one sequential colormap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for color, lr in zip(colors, SWEEP_LRS):
    h = sweep_histories[lr]
    if h['iter_loss']:
        ax1.plot(smooth(h['iter_loss']), color=color, lw=2, label=f'lr = {lr:g}')
ax1.axhline(math.log(NUM_CLASSES), color='gray', ls='--', lw=1)
ax1.text(0, math.log(NUM_CLASSES) * 1.04, 'ln(10): random guessing', color='gray', fontsize=9)
ax1.set_yscale('log')
ax1.set_xlabel('Training step'); ax1.set_ylabel('Training loss (smoothed, log scale)')
ax1.set_title('Training loss during the sweep'); ax1.legend(); ax1.grid(alpha=0.3)

final_acc = [sweep_histories[lr]['val_acc'][-1] if sweep_histories[lr]['val_acc'] else np.nan
             for lr in SWEEP_LRS]
ax2.plot(SWEEP_LRS, final_acc, 'o-', lw=2, markersize=8)
for lr, acc in zip(SWEEP_LRS, final_acc):
    if np.isnan(acc):
        ax2.annotate('diverged', (lr, 0.05), ha='center', color='gray')
ax2.set_xscale('log'); ax2.set_ylim(0, 1)
ax2.set_xlabel('Learning rate (log scale)'); ax2.set_ylabel('Validation accuracy after 3 epochs')
ax2.set_title('Learning-rate sweep'); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Questions

- Which curves match your predictions? For each learning rate, would you call it *too low*, *about
  right* or *too high*? What in the curve tells you?
- Was the baseline learning rate we used in Part 2 (`BASELINE_LR = 0.01`) a sensible choice?
- The best learning rate after 3 epochs is not necessarily the best after 30. Why might a slightly
  lower learning rate win in a longer run?
- We picked the learning rate using the **validation** set. Why would it be wrong to use the test set?

**Batch size** is often tuned next, but be careful: batch size and learning rate interact (a larger
batch gives less noisy gradients and usually tolerates, or needs, a larger learning rate). So
changing the batch size alone is not really a one-factor change. If you tune it in your project,
re-check the learning rate as well.

---
## Part 4: A worked iteration — why won't the deep network train?

This part walks through one complete trip around the loop, using the tools from this week's lecture.

**The situation:** you want more capacity than `ToyCNN`, so you build `DeepCNN` (9 conv layers).
You initialise the weights with small random numbers, N(0, 0.01²) — the "small random values" from
the lecture — and train it with exactly the same setup as the baseline.

### Step 1 — Clarification: what do we observe?

In [ ]:
for seed in [0, 1]:
    deep_small, deep_small_hist = run_experiment('E1_deep_small_init', arch='deep', init='small',
                                                 lr=BASELINE_LR, seed=seed)
summarise(['E0_baseline', 'E1_deep_small_init'])

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(smooth(deep_small_hist['iter_loss']), lw=2)
plt.axhline(math.log(NUM_CLASSES), color='gray', ls='--', lw=1, label='ln(10)')
plt.xlabel('Training step'); plt.ylabel('Training loss (smoothed)')
plt.title('DeepCNN, small init'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

print('Validation predictions by class:',
      torch.bincount(deep_small(x_val).argmax(1), minlength=NUM_CLASSES).tolist())
print('Most common class in training set: ', torch.bincount(y_train).argmax().item(),
      f'({torch.bincount(y_train).max().item() / len(y_train):.1%} of images)')

The deeper model does far *worse* than the small baseline. The loss falls a little below
ln(10) ≈ 2.30 and then stops.

**Question:** compare the validation accuracy with the share of the most common class in the
training set, and look at which classes the model predicts. What has the model actually learned?
(Hint: which parameters can still learn if everything before the last layer outputs almost nothing?)

So: **issue = the deep network doesn't learn.** It is the only issue, so prioritisation is easy
this time.

### Step 2 — Analysis: find possible causes

"It doesn't train" has many possible causes: a learning rate that's too low, a bug, too little
data, vanishing gradients, dead ReLUs… Guessing is slow. Instead, **look inside the network**
with two tools from the lecture:

1. **Activation statistics** — histograms of each layer's outputs. In a healthy network, the
   activations keep a similar spread from layer to layer. If they shrink towards zero (or blow up)
   as you go deeper, the signal is vanishing (or exploding).
2. **Gradient flow** — the average size of the gradient of the loss with respect to each layer's
   weights. If it is tiny, or shrinks by orders of magnitude towards the input, those layers barely learn.

We also track the **fraction of dead ReLU units**: channels that output exactly zero for *every*
image in the batch.

These three functions are reusable. They work for any `nn.Module` — including a U-Net or a
detection backbone — so copy them into your project.

In [ ]:
@torch.no_grad()
def activation_stats(model, x, layer_type=nn.ReLU):
    '''Records the output of every `layer_type` layer for the batch x. Returns a list of (name, tensor).'''
    # We use a copy in train mode, so that BN layers normalise with the batch statistics (as they do
    # during training) and their running statistics in the real model are not changed.
    model = copy.deepcopy(model).train()
    records, hooks = [], []
    for name, module in model.named_modules():
        if isinstance(module, layer_type):
            hooks.append(module.register_forward_hook(
                lambda m, inp, out, name=name: records.append((name, out.detach().float().cpu()))))
    model(x)
    for h in hooks:
        h.remove()
    return records


def plot_activation_histograms(model, x, title=''):
    '''One histogram per ReLU layer, plus mean/std and the fraction of dead units.'''
    records = activation_stats(model, x)
    n = len(records)
    fig, axes = plt.subplots(1, n, figsize=(1.9 * n, 2.6), sharey=True)
    for i, (ax, (name, act)) in enumerate(zip(axes, records)):
        vals = act.flatten().numpy()
        # A channel is "dead" if it is zero for every image and every position in the batch
        per_channel_max = act.transpose(0, 1).reshape(act.shape[1], -1).max(dim=1).values if act.dim() == 4 \
            else act.max(dim=0).values
        dead = (per_channel_max <= 0).float().mean().item()
        ax.hist(vals, bins=40, color='steelblue')
        ax.set_yscale('log')
        ax.set_title(f'ReLU {i + 1}\nmean {vals.mean():.2g}\nstd {vals.std():.2g}\ndead {dead:.0%}', fontsize=8)
        ax.tick_params(labelsize=7)
    axes[0].set_ylabel('count (log)')
    plt.suptitle(f'Activation histograms, input → output  {title}', y=1.3)
    plt.show()


def plot_grad_flow(model, x, y, title=''):
    '''Mean absolute gradient of the loss w.r.t. the weights of each conv/linear layer, for one batch.'''
    model.train()
    model.zero_grad()
    loss_fn(model(x), y).backward()
    names, grads = [], []
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            names.append(name)
            grads.append(module.weight.grad.abs().mean().item())
    model.zero_grad()
    plt.figure(figsize=(7, 3))
    plt.bar(range(len(grads)), grads, color='coral')
    plt.yscale('log')
    plt.xticks(range(len(grads)), [f'{i + 1}' for i in range(len(grads))])
    plt.xlabel('Layer (input → output)')
    plt.ylabel('mean |∂loss/∂W| (log scale)')
    plt.title(f'Gradient flow  {title}')
    plt.grid(alpha=0.3, axis='y')
    plt.show()
    return grads


x_probe, y_probe = x_train[:256], y_train[:256]   # a fixed probe batch, reused for every diagnosis below

✍️ **Prediction:** for the DeepCNN with small initialisation, what will the activation histograms
look like as you go deeper? And the gradients?

We look at the network **at initialisation** (before training), because that's where the problem
begins.

In [ ]:
set_seed(0)
untrained_small = build_model('deep', init='small')
plot_activation_histograms(untrained_small, x_probe, '(DeepCNN, small init, untrained)')
g_small = plot_grad_flow(untrained_small, x_probe, y_probe, '(DeepCNN, small init, untrained)')
print(f'Largest weight gradient in any layer: {max(g_small):.1e}')

**Questions:**

- Describe what happens to the activation standard deviation from layer to layer. By roughly what
  factor does it shrink per layer?
- How does this connect to the lecture's explanation of repeated matrix multiplications and
  vanishing/exploding values?
- The weight gradients are tiny in *every* layer, not only near the input. Why? (Recall
  backpropagation: the gradient of a layer's weights is its *input activation* times the *error*
  propagated back to it. Which of the two is small near the input, and which near the output?)
- Is the problem dead ReLUs, or something else?

### Step 3 — Design solutions: hypotheses

From the analysis, the most likely cause is that **the weights are too small, so the signal shrinks
at every layer** until nothing is left. Two candidate fixes from the lecture:

- **H1 — Initialisation.** Kaiming initialisation scales each layer's weights (std = √(2 / fan_in))
  so that the activation variance stays roughly constant through ReLU layers.
  *Single change:* `init='small'` → `init='kaiming'`.
- **H2 — Batch normalisation.** BN re-normalises every layer's outputs to zero mean and unit
  variance (then applies a learnable scale γ and shift β), so the signal can't shrink regardless of
  the weight scale. *Single change:* `batchnorm=False` → `batchnorm=True`, **keeping the small init**.

✍️ **Prediction:** write it down before running.

| | Will it train? | Val. accuracy vs. E0 baseline? | What will the histograms look like? |
|---|---|---|---|
| H1 (Kaiming) | | | |
| H2 (BN, small init) | | | |

### Step 4 — Evaluation: test each hypothesis, one change at a time

In [ ]:
for seed in [0, 1]:
    deep_kaiming, deep_kaiming_hist = run_experiment('E2_deep_kaiming', arch='deep', init='kaiming',
                                                     lr=BASELINE_LR, seed=seed)
for seed in [0, 1]:
    deep_bn, deep_bn_hist = run_experiment('E3_deep_small_init+BN', arch='deep', init='small', batchnorm=True,
                                           lr=BASELINE_LR, seed=seed)

summarise(['E0_baseline', 'E1_deep_small_init', 'E2_deep_kaiming', 'E3_deep_small_init+BN'])

In [ ]:
set_seed(0)
untrained_kaiming = build_model('deep', init='kaiming')
plot_activation_histograms(untrained_kaiming, x_probe, '(DeepCNN, Kaiming init, untrained)')
plot_grad_flow(untrained_kaiming, x_probe, y_probe, '(DeepCNN, Kaiming init, untrained)')

set_seed(0)
untrained_bn = build_model('deep', init='small', batchnorm=True)
plot_activation_histograms(untrained_bn, x_probe, '(DeepCNN, small init + BN, untrained)')
plot_grad_flow(untrained_bn, x_probe, y_probe, '(DeepCNN, small init + BN, untrained)');

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
runs = [('E1 small init', deep_small_hist), ('E2 Kaiming', deep_kaiming_hist), ('E3 small init + BN', deep_bn_hist)]
for label, h in runs:
    ax1.plot(smooth(h['iter_loss']), lw=2, label=label)
    ax2.plot(range(1, len(h['val_acc']) + 1), h['val_acc'], 'o-', lw=2, label=label)
ax1.set_xlabel('Training step'); ax1.set_ylabel('Training loss (smoothed)'); ax1.set_title('Training loss (seed 1)')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Validation accuracy'); ax2.set_title('Validation accuracy (seed 1)')
for ax in (ax1, ax2):
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Questions:**

- Were your predictions right? Which hypothesis is supported by the evidence? Both? Neither?
- Look at the histograms again. The two fixes solve the same problem in different ways — explain
  the difference.
- Does the deep model now beat the `ToyCNN` baseline by more than the noise floor from Part 2?
  If it doesn't, what does that tell you about the value of extra depth *on this subset and
  training budget*?
- Compare the final **training** loss with the **validation** loss for E0, E2 and E3 (in the
  summary table and the log). What does the gap tell you? Which issue would you put at the top of
  the list for the next iteration of the loop?
- Write **one sentence** for a Results section and **one sentence** for a Discussion section
  about this experiment (see the Week 4 worksheet for the difference).

### A batch-norm gotcha: `model.train()` vs. `model.eval()`

BN behaves differently during training and evaluation. In `train()` mode it normalises with the
statistics of the **current batch**. In `eval()` mode it uses **running averages** collected during
training. So in train mode, the prediction for one image depends on *which other images are in
the same batch*. Forgetting `model.eval()` is a common source of confusing results.

We evaluate the trained BN model from Step 4 in both modes, with three kinds of batches:
- **shuffled, batch size 256:** a random mix of classes (like the training batches);
- **shuffled, batch size 2:** only two images per batch;
- **sorted by class, batch size 64:** each batch contains (almost) only one digit. This happens easily
  in practice, e.g. with an unshuffled evaluation loader over data stored one folder per class, as
  with `ImageFolder` in Lab 3.

✍️ **Prediction:** in which of the six cases will accuracy change noticeably from the eval-mode value? Why?

In [ ]:
@torch.no_grad()
def accuracy_in_mode(model, mode, batch_size, sort_by_class=False):
    m = copy.deepcopy(model)          # a copy: in train mode, BN would also update its running statistics
    m.train() if mode == 'train' else m.eval()
    order = torch.argsort(y_val) if sort_by_class else torch.randperm(len(y_val), device=y_val.device)
    xs, ys = x_val[order], y_val[order]
    correct = 0
    for xb, yb in batches(xs, ys, batch_size=batch_size, shuffle=False):
        correct += (m(xb).argmax(1) == yb).sum().item()
    return correct / len(ys)


torch.manual_seed(0)
for label, bs, sort in [('shuffled, batch size 256', 256, False),
                        ('shuffled, batch size   2', 2, False),
                        ('sorted by class, batch size 64', 64, True)]:
    print(f'{label:<32} eval mode {accuracy_in_mode(deep_bn, "eval", bs, sort):.3f}  |  '
          f'train mode {accuracy_in_mode(deep_bn, "train", bs, sort):.3f}')

**Why doesn't batch size 2 hurt much here?** In a *convolutional* layer, BN computes each channel's
statistics over the batch **and all spatial positions**, so even two 32×32 images give hundreds to
thousands of values per channel. The statistics are still reasonable. What breaks BN is not a small
batch as such, but a batch that is *not representative* of the data, as in the class-sorted case.

**Why this matters for your project:** most of you will fine-tune a pretrained network (ResNet,
EfficientNet, a U-Net encoder…) that contains BN layers. If your images are large and your GPU
memory only allows batches of 2–8, the batch statistics during fine-tuning become very noisy.
A common remedy is to keep the pretrained BN layers in eval mode (frozen running statistics) while
fine-tuning the rest. If you do this, say so in your Methods section — it is exactly the kind of
motivated design decision a good report explains.

---
## Part 5: Ablation — what does each component contribute?

Parts 2–4 followed an **experiment ladder**: start from the baseline and *add* one change at a
time. An **ablation study** goes the other way: start from your *final* model and *remove* one
component at a time, to show how much each component contributes to the final result.

Say our final model is DeepCNN with **both** Kaiming initialisation and BN. We already have most
of the runs we need. Only the full model is missing.

✍️ **Prediction:** fill in the table before running.

| Model | Predicted val. accuracy |
|---|---|
| Full: Kaiming + BN | |
| − BN (= E2) | |
| − Kaiming (= E3) | |
| − both (= E1) | |

In [ ]:
for seed in [0, 1]:
    deep_full, _ = run_experiment('E4_deep_kaiming+BN', arch='deep', init='kaiming', batchnorm=True,
                                  lr=BASELINE_LR, seed=seed)

ablation = summarise(['E4_deep_kaiming+BN', 'E2_deep_kaiming', 'E3_deep_small_init+BN', 'E1_deep_small_init'])
ablation.index = ['Full: Kaiming + BN', '− BN', '− Kaiming', '− both']
ablation

**Questions:**

- In the *ladder* (Part 4), switching to Kaiming initialisation made a huge difference. In the
  *ablation*, how much does removing Kaiming from the full model cost? Why are these two answers
  different?
- This is an **interaction**: the effect of one component depends on whether another is present.
  Why does this make it misleading to report only the ladder, or only the ablation?
- Your report should usually include both: the ladder shows the *story* of how you got to the final
  model, and the ablation shows *what the final model depends on*.

- Look at the training loss of the full model. What issue does it reveal, and which techniques
  (some from next week's lecture) could address it?

**Extension (if you have time):** add data augmentation (`augment=True`) to the full model as a
new rung on the ladder. Write your prediction first. After 10 epochs, does it help? Look at the gap
between training and validation loss for both runs — what does augmentation do to it, and what might
happen with a longer training budget?

---
## Part 6: Your experiment budget

Every experiment above took seconds. Your project's runs will probably take minutes or hours —
and the number of runs you can afford decides how many questions you can answer. Groups that
don't plan this end up in late November with one baseline and one "improvement" run, and nothing
to compare them against.

Look at the runtimes in the log:

In [ ]:
log = pd.DataFrame(EXPERIMENT_LOG)
log['sec_per_epoch'] = log['runtime_s'] / log['epochs']
print(log.groupby('arch')['sec_per_epoch'].mean().round(2).rename('seconds per epoch'))
print(f'\nTotal runs so far: {len(log)},  total training time: {log["runtime_s"].sum() / 60:.1f} min')

### Estimate your own budget (do this for your project in the worksheet)

```
runs you can afford ≈ (GPU hours available until the deadline × 3600)
                      / (seconds per epoch × epochs per run)
```

Then subtract seeds: if every configuration needs 3 seeds, divide by 3.

**If the number is too small, make each run cheaper before you start the real experiments:**

| Technique | Typical speed-up | Caveat |
|---|---|---|
| Screen on a data subset (like today) | proportional to the subset size | Confirm the final comparisons on the full data |
| Lower input resolution (e.g., 512 → 224) | ~5× for 512 → 224 | Small objects/details may disappear — check visually |
| Fewer epochs for screening runs | proportional | Rankings can change in longer runs (Part 3) |
| Pre-compute / cache preprocessing | often large when loading is the bottleneck | Only for deterministic steps (not augmentation) |
| Freeze the backbone, train only the head | large | It's a different experiment — log it as such |

**Protect your runs from Colab disconnects:** save a checkpoint every epoch to Google Drive, so a
disconnect costs you one epoch rather than the whole run:

In [ ]:
def save_checkpoint(path, model, optimizer, epoch, config):
    torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                'epoch': epoch, 'config': config}, path)


def load_checkpoint(path, model, optimizer):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    return ckpt['epoch'], ckpt['config']

# In Colab:
# from google.colab import drive; drive.mount('/content/drive')
# save_checkpoint('/content/drive/MyDrive/project/ckpt_E2_seed0.pt', model, optimizer, epoch, config)

---
## Part 7 (optional): Is the model looking at the right thing? — Grad-CAM

A model can be right for the wrong reasons: it might use the background, a watermark, or a
hospital's scanner label instead of the object itself (**shortcut learning**). Accuracy can't
reveal this. **Grad-CAM** produces a heat map of the image regions that most influenced the
prediction for a given class.

Grad-CAM is covered properly in Lecture 10 — **you don't need to understand how it works yet**.
Treat `grad_cam` below as a black-box tool.

SVHN has a built-in shortcut risk: each image is centred on one digit, but neighbouring digits are
often visible at the edges (Lab 4a). So a natural question is whether the model looks at the
**centre digit** or at its neighbours.

In [ ]:
def grad_cam(model, target_layer, x, class_idx=None):
    '''Returns (heatmaps in [0, 1] of shape (N, H, W), explained class indices). Black box for now — Lecture 10.'''
    store = {}

    def forward_hook(module, inp, out):
        store['act'] = out
        out.register_hook(lambda grad: store.__setitem__('grad', grad))

    handle = target_layer.register_forward_hook(forward_hook)
    model.eval()
    with torch.enable_grad():
        logits = model(x)
        if class_idx is None:
            class_idx = logits.argmax(1)
        model.zero_grad()
        logits.gather(1, class_idx.view(-1, 1)).sum().backward()
    handle.remove()
    weights = store['grad'].mean(dim=(2, 3), keepdim=True)              # importance of each channel
    cam = F.relu((weights * store['act']).sum(dim=1, keepdim=True))     # weighted sum of feature maps
    cam = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)
    cam = cam / (cam.amax(dim=(2, 3), keepdim=True) + 1e-8)
    return cam.squeeze(1).detach().cpu(), class_idx.cpu()


# Explain the last ReLU of the feature extractor (8x8 resolution) in the full model from Part 5
last_relu = [m for m in deep_full.features if isinstance(m, nn.ReLU)][-1]
with torch.no_grad():
    preds = deep_full.eval()(x_val).argmax(1)
correct_idx = torch.where(preds == y_val)[0][:6]
wrong_idx = torch.where(preds != y_val)[0][:6]

fig, axes = plt.subplots(2, 6, figsize=(14, 5.2))
for row, (label, idx) in enumerate([('correct', correct_idx), ('wrong', wrong_idx)]):
    cams, cls = grad_cam(deep_full, last_relu, x_val[idx])
    imgs = (x_val[idx].cpu() * STD + MEAN).clamp(0, 1)
    for col in range(len(idx)):
        ax = axes[row, col]
        ax.imshow(imgs[col].permute(1, 2, 0).numpy())
        ax.imshow(cams[col].numpy(), cmap='jet', alpha=0.4)
        ax.set_title(f'{label}: true {y_val[idx[col]].item()}, pred {cls[col].item()}', fontsize=9)
        ax.axis('off')
plt.suptitle('Grad-CAM for the predicted class (red = most influential region)')
plt.tight_layout()
plt.show()

**Questions:**

- For the correct predictions, does the model focus on the centre digit?
- For the wrong predictions, can the heat map explain the mistake — is the model looking at a
  neighbouring digit, for example?
- Is there a potential shortcut in **your** project's data (backgrounds, text overlays, image
  borders, acquisition-site differences)? If so, Grad-CAM (or simply cropping or occluding the
  suspicious region and re-evaluating) is a way to test it. Note it in your iteration log.

---
## Wrapping up: your toolbox

Functions from today that you can copy straight into your project:

| Function | Use it when |
|---|---|
| `check_data`, `check_split`, `check_initial_loss`, `overfit_one_batch` | **Every** new pipeline, and after every major code change |
| `run_experiment` + `EXPERIMENT_LOG` (adapt to your task) | Every run — log the config and the results automatically |
| `summarise` | Comparing configurations: mean ± std over seeds |
| `activation_stats`, `plot_activation_histograms`, `plot_grad_flow` | The model doesn't train, trains slowly, or is unstable |
| `save_checkpoint`, `load_checkpoint` | Any run longer than a few minutes |
| `grad_cam` | You suspect the model is right for the wrong reasons |

## Now: your own project

Open **`Lab6_IterationLog.md`** with your group and:

1. Run the four sanity checks on **your** pipeline (Part A of the log).
2. Run your baseline with at least 2–3 seeds if you can, or plan when you will.
3. Estimate your experiment budget (Part B).
4. Complete your first iteration page (Part C) and discuss it with a TA before you leave.